# Deep Learning Volatility Forecasting V5-Sens (Sensitivity Run: Patience=20, Kausal, Terstandarisasi)
Notebook ini adalah varian sensitivity run dari V5 yang dirancang khusus untuk menguji ketahanan konvergensi model terhadap ambang batas early stopping yang lebih longgar (`patience=20` validation checks).

### Parameter & Modifikasi Khusus V5S:
1. **Early Stopping Patience Longgar**: `CHOSEN_PATIENCE = 20` checks x `val_check_steps = 25` = 500 steps (<= 1000 max_steps).
2. **Konfigurasi Terfokus**: 4 konfigurasi model (TFT & NBEATSx x {SPILL, NOSPILL} pada target T0).
3. **Nama Berkas Khusus**: Seluruh luaran menggunakan awalan `v5s_` (`v5s_partial_*.csv`, `v5s_metrics_table.csv`, `v5s_dm_table.csv`, `v5s_es_log.csv`, `v5s_outputs.zip`, `v5s_md5_manifest.txt`).
4. **Audit State Callback Internal**: Logging transparan `stopped_epoch`, `best_score`, `wait_count`, dan `patience` ke `v5s_es_log.csv` tepat setelah fit.
5. **TFT Importance Transparan Penuh**: Menampilkan seluruh 20 fitur dengan label eksplisit `window terakhir train+val, input_size=50, n_cutoffs=1`.
6. **Stage A & B Konsisten**: Validasi integritas data, benchmark HAR v2, dan smoke test tetap aktif dengan strict assertion yang identik.


# CELL 1: SETUP LINGKUNGAN KAGGLE, GUARD, PINNING, SEED, & LOGGING


In [1]:

import os
import sys
import glob
import time
import json
import random
import socket
import zipfile
import hashlib
import logging
import warnings
import subprocess
import numpy as np
import pandas as pd
import torch

# 1. Deteksi GPU yang Tersedia (Patch 4)
print("=== 1. DETEKSI PERANGKAT GPU ===")
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU Terdeteksi: {gpu_name} (Total: {gpu_count} GPU)")
else:
    print("⚠️ GPU TIDAK TERDETEKSI! Notebook berjalan dalam mode CPU.")

# 2. Cek Koneksi Internet Sebelum Pip Install (Patch 4)
def check_internet(host="8.8.8.8", port=53, timeout=3):
    try:
        socket.setdefaulttimeout(timeout)
        socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect((host, port))
        return True
    except (socket.timeout, socket.error):
        return False

print("\n=== 2. KONEKSI INTERNET & PINNING DEPENDENCY ===")
has_net = check_internet()
print(f"Status Koneksi Internet: {'TERHUBUNG (Online)' if has_net else 'TERPUTUS (Offline)'}")

# 3. Pip Install Pin NeuralForecast 3.3.0 di Kaggle (Item 2)
try:
    import neuralforecast
    nf_ver = neuralforecast.__version__
except ImportError:
    nf_ver = None

if has_net and (os.path.exists('/kaggle') or (nf_ver != '3.3.0') or (nf_ver is None)):
    print("Menginstal/menyelaraskan paket neuralforecast==3.3.0...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuralforecast==3.3.0', 'arch', 'pytorch-lightning'], check=False)
elif not has_net and (nf_ver is None or nf_ver != '3.3.0'):
    print("⚠️ Peringatan: Tidak ada internet untuk pip install, pastikan paket telah tersedia di lingkungan.")

try:
    import neuralforecast
    print(f"Versi NeuralForecast terpasang: {neuralforecast.__version__}")
except Exception as e:
    print(f"Peringatan impor neuralforecast: {e}")

from neuralforecast import NeuralForecast
from neuralforecast.models import TFT, TiDE, NBEATSx, TSMixerx, KAN, NHITS, LSTM

# 4. Kaggle Guard: Deteksi Folder Input (Patch 4)
print("\n=== 3. KAGGLE INPUT GUARD ===")
DATASET_DIR = '/kaggle/input/datasets/ferastoro/spillover-v5-inputs/rsic-v5-inputs'
if not (os.path.exists(DATASET_DIR) and os.path.exists(os.path.join(DATASET_DIR, 'dataset_deep_learning.csv'))):
    if os.path.exists('/kaggle/input'):
        input_matches = glob.glob('/kaggle/input/**/dataset_deep_learning.csv', recursive=True)
        candidate_dirs = sorted(list(set([os.path.dirname(m) for m in input_matches])))
        print(f"Direktori kandidat di /kaggle/input: {candidate_dirs}")
        assert len(candidate_dirs) == 1, (
            f"KAGGLE GUARD FAIL: Ditemukan {len(candidate_dirs)} folder yang memuat dataset_deep_learning.csv di /kaggle/input! "
            f"Harus tepat 1 folder. Daftar folder: {candidate_dirs}"
        )
        DATASET_DIR = candidate_dirs[0]
    else:
        # Local fallback (auto-detect inputs/v5 or ../inputs/v5)
        BASE_DIR = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.')
        candidates = [
            os.path.join(BASE_DIR, 'inputs', 'v5'),
            'inputs/v5',
            '../inputs/v5',
            'rsic-v5-inputs',
            '../rsic-v5-inputs'
        ]
        DATASET_DIR = '.'
        for cand in candidates:
            if os.path.exists(os.path.join(cand, 'dataset_deep_learning.csv')):
                DATASET_DIR = cand
                break

print(f"✅ DATASET_DIR Terpilih: {DATASET_DIR}")
dir_contents = sorted(os.listdir(DATASET_DIR)) if os.path.exists(DATASET_DIR) else []
print(f"Isi Folder DATASET_DIR ({len(dir_contents)} item):")
for item in dir_contents:
    item_p = os.path.join(DATASET_DIR, item)
    sz = os.path.getsize(item_p) if os.path.isfile(item_p) else 0
    print(f"  - {item:<36} ({sz/1024:8.1f} KB)")

if DATASET_DIR not in sys.path:
    sys.path.append(DATASET_DIR)
SRC_DIR = os.path.join(BASE_DIR, 'src') if 'BASE_DIR' in locals() else os.path.abspath('../src')
if os.path.exists(SRC_DIR) and SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

OUTPUT_DIR = '/kaggle/working' if os.path.exists('/kaggle') else 'scratch/final'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Fungsi Backup ZIP Otomatis (P7)
def save_zip_backup():
    zip_path = os.path.join(OUTPUT_DIR, 'v5s_outputs.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in glob.glob(os.path.join(OUTPUT_DIR, 'v5s_*')):
            if not f.endswith('.zip'):
                zf.write(f, os.path.basename(f))
    return zip_path

# 5. Global Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 6. Logging & IOPub Suppression
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
logging.getLogger("lightning").setLevel(logging.WARNING)
logging.getLogger("neuralforecast").setLevel(logging.WARNING)
warnings.filterwarnings('ignore')

# 7. Cetak Versi & Simpan Pip Freeze
print("\n=== 4. VERSI RUNTIME & PIP FREEZE ===")
print(f"Python: {sys.version.split()[0]}")
print(f"Torch: {torch.__version__}")
print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")

freeze_path = os.path.join(OUTPUT_DIR, "v5s_pip_freeze.txt")
try:
    res_freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True)
    with open(freeze_path, 'w') as f:
        f.write(res_freeze.stdout)
    print(f"Pip freeze tersimpan ke: {freeze_path}")
except Exception as e:
    print(f"Peringatan: Gagal menyimpan pip freeze: {e}")


=== 1. DETEKSI PERANGKAT GPU ===
✅ GPU Terdeteksi: Tesla T4 (Total: 2 GPU)

=== 2. KONEKSI INTERNET & PINNING DEPENDENCY ===
Status Koneksi Internet: TERHUBUNG (Online)
Menginstal/menyelaraskan paket neuralforecast==3.3.0...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.3/331.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 982.9/982.9 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.7/348.7 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 3.5 MB/s eta 0:00:00
Versi NeuralForecast terpasang: 3.3.0

=== 3. KAGGLE INPUT GUARD ===
✅ DATASET_DIR Terpilih: /kaggle/input/datasets/ferastoro/spillover-v5-inputs/rsic-v5-inputs
Isi Folder DATASET_DIR (8 item):
  - CV_Predictions_V4.csv                (   491.9 KB)
  - GARCH_Predictions_V4.csv             (   277.7 KB)
  - dataset_deep_learning.csv            (   451.1 KB)
  - har_preds_T0_h

# CELL 2: KONFIGURASI STAGE, EARLY STOPPING A PRIORI, & ABLATION CONFIGS


In [2]:
# P4: Konfigurasi Training & Early Stopping Terpicu A Priori (Sensitivity V5S)
# Aritmetika: 20 validation checks x 25 val_check_steps = 500 steps <= 1000 max_steps.
# Nilai dipilih a priori (bukan dari hasil test) untuk menjamin validitas konvergensi.
VAL_CHECK_STEPS = 25
CHOSEN_PATIENCE = 20     # 20 checks x 25 steps/check = 500 steps total
MAX_STEPS = 1000
H = 10                    # Horizon peramalan (h=1 s.d. 10)
INPUT_SIZE = 5 * H        # Input lag window = 50
REFIT_PER_CUTOFF = False  # Standar DL rolling inference

# Opsi STAGE: "A" (Audit CPU), "B" (Smoke Test 3 Cutoff), "C" (Training Penuh)
STAGE = "C"  # <--- GANTI DI SINI: "A", "B", atau "C"

# P5: Feature Sets untuk Ablation Study
# 1. NOSPILL (11 fitur): 8 return/weekend + 3 kalender
NOSPILL_COLS = [
    'ret_saham_nvda', 'ret_saham_msft', 'ret_saham_tsla', 'ret_saham_googl', 'ret_saham_meta',
    'ret_crypto_btc_usd', 'ret_crypto_eth_usd', 'crypto_btc_usd_weekend_return',
    'gap_days', 'is_monday', 'is_ihsg_holiday'
]

# 2. SPILL (20 fitur): 11 NOSPILL + 9 spillover kausal murni
SPILL_COLS = NOSPILL_COLS + [
    'spillover_ai_to_btcidr', 'tvp_tsi',
    'net_spill_saham_nvda', 'net_spill_saham_msft', 'net_spill_saham_tsla', 'net_spill_saham_googl', 'net_spill_saham_meta',
    'net_spill_crypto_btc_usd', 'net_spill_crypto_eth_usd'
]

# 3. VOLX (18 fitur): 11 NOSPILL + log(r2+1e-10) dari 7 aset pada hari t
VOLX_ASSETS = ['ret_saham_nvda', 'ret_saham_msft', 'ret_saham_tsla', 'ret_saham_googl', 'ret_saham_meta', 'ret_crypto_btc_usd', 'ret_crypto_eth_usd']
VOLX_COLS = NOSPILL_COLS + [f'volx_{a}' for a in VOLX_ASSETS]

FEATURE_SETS = {
    'SPILL': SPILL_COLS,
    'NOSPILL': NOSPILL_COLS,
    'VOLX': VOLX_COLS
}

# Single Configuration List (Sensitivity Run V5S)
# Hanya TFT dan NBEATSx x {SPILL, NOSPILL} T0 saja (4 konfigurasi)
CONFIGS = [
    ('TFT', 'SPILL', 'T0'),
    ('TFT', 'NOSPILL', 'T0'),
    ('NBEATSx', 'SPILL', 'T0'),
    ('NBEATSx', 'NOSPILL', 'T0')
]

print(f"Konfigurasi aktif: STAGE={STAGE}")
print(f"Aritmetika Early Stopping: patience={CHOSEN_PATIENCE} checks x val_check={VAL_CHECK_STEPS} steps = {CHOSEN_PATIENCE * VAL_CHECK_STEPS} steps <= max_steps={MAX_STEPS}")
print(f"Daftar Konfigurasi Model & Feature Set ({len(CONFIGS)} item):")
for cfg in CONFIGS:
    print(f"  - Model: {cfg[0]:<10} | Feature Set: {cfg[1]:<10} ({len(FEATURE_SETS[cfg[1]])} feat) | Target: {cfg[2]}")


Konfigurasi aktif: STAGE=C
Aritmetika Early Stopping: patience=20 checks x val_check=25 steps = 500 steps <= max_steps=1000
Daftar Konfigurasi Model & Feature Set (4 item):
  - Model: TFT        | Feature Set: SPILL      (20 feat) | Target: T0
  - Model: TFT        | Feature Set: NOSPILL    (11 feat) | Target: T0
  - Model: NBEATSx    | Feature Set: SPILL      (20 feat) | Target: T0
  - Model: NBEATSx    | Feature Set: NOSPILL    (11 feat) | Target: T0


# CELL 3: PEMUATAN DATA, MERGE TVP KAUSAL, SCHEMA, & SPLITS


In [3]:
import hashlib

def get_file_md5(p):
    if not os.path.exists(p):
        return "NOT_FOUND"
    h = hashlib.md5()
    with open(p, 'rb') as fp:
        while chunk := fp.read(8192):
            h.update(chunk)
    return h.hexdigest()

# P2a: expected_hashes diperbarui ke file _v2 (legacy files dihapus dari daftar)
expected_hashes = {
    'dataset_deep_learning.csv': 'ba64297407f9d8221488b4815a12ce8c',
    'tvp_var_dynamic_features_causal.csv': '066ad3cdf980cd187a79c25dc210f1b2',
    'v5_helpers.py': 'bdd7bf54c3cca56f30a7a1cd779f65b1',
    'har_preds_T0_h1_v2.csv': 'e340a7ba8effd60c8c6d7b85b073aded',
    'har_preds_T0_h10_v2.csv': '43baf149f5f60be52a80b962ab93e481',
    'CV_Predictions_V4.csv': '6251bfb6ce543ffcd746e5fab98c3a63',
    'GARCH_Predictions_V4.csv': '5e84fa827fd5cfd13bea445230daf2ee'
}

print("=== AUDIT MD5 FILE INPUT (P2a) ===")
for fname, exp_h in expected_hashes.items():
    fpath = os.path.join(DATASET_DIR, fname)
    act_h = get_file_md5(fpath)
    status = "PASS" if act_h == exp_h else f"FAIL (Found: {act_h})"
    print(f"File {fname:<36} | MD5: {act_h} | Status: {status}")
    assert act_h == exp_h, f"ASSERT FAILED: Hash {fname} tidak cocok! (Harapan: {exp_h}, Aktual: {act_h})"

# 2. Pemuatan Dataset Primer
dl_file = os.path.join(DATASET_DIR, 'dataset_deep_learning.csv')
tvp_file = os.path.join(DATASET_DIR, 'tvp_var_dynamic_features_causal.csv')

df_dl = pd.read_csv(dl_file)
df_tvp = pd.read_csv(tvp_file)

df_dl['Date'] = pd.to_datetime(df_dl['Date'])
df_tvp['Date'] = pd.to_datetime(df_tvp['Date'])

# Item 5: Satu hist_series dari df_dl (ret**2, index Date) dipakai di Cell 4, 7, 8
hist_series = pd.Series(df_dl['ret_crypto_btc_idr_indodax'].values**2, index=df_dl['Date'])

# 3. Merge Causal TVP Features
df_merged = pd.merge(df_dl, df_tvp, on='Date', how='left').sort_values('Date')
tvp_cols = [c for c in df_tvp.columns if c != 'Date']
df_merged[tvp_cols] = df_merged[tvp_cols].ffill()
df = df_merged.dropna().reset_index(drop=True)

# 4. Target T0 & T2
df['volatility_btc_idr'] = df['ret_crypto_btc_idr_indodax'] ** 2  # Target T0 (r2_1)

alt_file = os.path.join(DATASET_DIR, 'target_alternatives.csv')
if os.path.exists(alt_file):
    alt_df = pd.read_csv(alt_file)
    alt_df['Date'] = pd.to_datetime(alt_df['Date'])
    df = pd.merge(df, alt_df[['Date', 'parkinson']], on='Date', how='left').reset_index(drop=True)
    df['parkinson'] = df['parkinson'].ffill()
    p_min = df.loc[df['parkinson'] > 0, 'parkinson'].min()
    df.loc[df['parkinson'] == 0, 'parkinson'] = p_min / 2.0
    print("✅ Target T2 (parkinson) berhasil dimuat dari target_alternatives.csv")
else:
    print("ℹ️ target_alternatives.csv tidak ada di DATASET_DIR. T2 akan dimuat jika file tersedia.")

# P5: Hitung Fitur VOLX log(r2 + 1e-10)
for a in VOLX_ASSETS:
    df[f'volx_{a}'] = np.log(df[a]**2 + 1e-10).astype('float32')

# 5. Verifikasi Jumlah Baris & Leakage
assert len(df) == 1676, f"Total baris df harus 1676! Ditemukan: {len(df)}"
assert len(SPILL_COLS) == 20, "Jumlah kolom SPILL harus tepat 20!"
assert len(NOSPILL_COLS) == 11, "Jumlah kolom NOSPILL harus tepat 11!"
assert len(VOLX_COLS) == 18, "Jumlah kolom VOLX harus tepat 18!"
assert 'net_spill_crypto_btc_idr_indodax' not in SPILL_COLS, "LEAKAGE: Target spillover tidak boleh ada di fitur eksogen!"
assert all(c in df.columns for c in SPILL_COLS), "Ada fitur SPILL yang hilang dari DataFrame!"
assert all(c in df.columns for c in VOLX_COLS), "Ada fitur VOLX yang hilang dari DataFrame!"

# 6. Bentuk Struktur Dataset Nixtla
nf_df = pd.DataFrame()
nf_df['ds'] = pd.date_range(start='2020-01-01', periods=len(df), freq='D')
nf_df['unique_id'] = 'BTC_IDR'
nf_df['y'] = df['volatility_btc_idr'].values.astype('float32')
all_exog_needed = sorted(list(set(SPILL_COLS + VOLX_COLS)))
for col in all_exog_needed:
    nf_df[col] = df[col].values.astype('float32')
nf_df['real_date'] = df['Date'].values

# 7. Split Kausal (1174 Train / 167 Val / 335 Test)
N = len(nf_df)
n_test = 335
n_val = 167
n_train = N - n_test - n_val

df_train = nf_df.iloc[:n_train].copy()
df_val = nf_df.iloc[n_train:n_train+n_val].copy()
df_test = nf_df.iloc[n_train+n_val:].copy()

print(f"Data siap: N={N} | Train={len(df_train)} (1174) | Val={len(df_val)} (167) | Test={len(df_test)} (335)")


=== AUDIT MD5 FILE INPUT (P2a) ===
File dataset_deep_learning.csv            | MD5: ba64297407f9d8221488b4815a12ce8c | Status: PASS
File tvp_var_dynamic_features_causal.csv  | MD5: 066ad3cdf980cd187a79c25dc210f1b2 | Status: PASS
File v5_helpers.py                        | MD5: bdd7bf54c3cca56f30a7a1cd779f65b1 | Status: PASS
File har_preds_T0_h1_v2.csv               | MD5: e340a7ba8effd60c8c6d7b85b073aded | Status: PASS
File har_preds_T0_h10_v2.csv              | MD5: 43baf149f5f60be52a80b962ab93e481 | Status: PASS
File CV_Predictions_V4.csv                | MD5: 6251bfb6ce543ffcd746e5fab98c3a63 | Status: PASS
File GARCH_Predictions_V4.csv             | MD5: 5e84fa827fd5cfd13bea445230daf2ee | Status: PASS
✅ Target T2 (parkinson) berhasil dimuat dari target_alternatives.csv
Data siap: N=1676 | Train=1174 (1174) | Val=167 (167) | Test=335 (335)


# CELL 4: STAGE A - VALIDASI PROVENANCE, HAR v2, STRICT DM ASSERT, & GARCH


In [4]:
import v5_helpers
from sklearn.metrics import mean_squared_error

print("=== STAGE A: AUDIT PROVENANCE & CROSS-CHECK BENCHMARK ===")

# 1. Cek Duplikat [cutoff, real_date] dari V4 & HAR
cv_v4_file = os.path.join(DATASET_DIR, 'CV_Predictions_V4.csv')
assert os.path.exists(cv_v4_file), f"File {cv_v4_file} tidak ditemukan!"
df_cv4 = pd.read_csv(cv_v4_file)
df_cv4['cutoff'] = pd.to_datetime(df_cv4['cutoff'])
df_cv4['real_date'] = pd.to_datetime(df_cv4['real_date'])
dup_count = df_cv4.duplicated(subset=['cutoff', 'real_date']).sum()
print(f"Duplikat [cutoff, real_date] di CV_Predictions_V4: {dup_count} (PASS: harus 0)")
assert dup_count == 0, "Ditemukan duplikasi baris pada cutoff dan real_date!"

# 2. Hitung Baseline Evaluasi (v5_helpers)
# Item 5: Menggunakan hist_series dari Cell 3 (1677 baris, ret**2, index Date)
cutoff_origins = v5_helpers.get_cutoff_origins(df_cv4, hist_series)
assert len(cutoff_origins) == 335, f"Jumlah cutoff harus 335, ditemukan {len(cutoff_origins)}"

grp_h1 = df_cv4.groupby('cutoff').head(1)
grp_h10 = df_cv4.groupby('cutoff').tail(1)

hexp_h1, roll_h1, persist_h1 = v5_helpers.compute_baselines(grp_h1, hist_series, cutoff_origins)
hexp_h10, roll_h10, persist_h10 = v5_helpers.compute_baselines(grp_h10, hist_series, cutoff_origins)

mse_hexp_1 = mean_squared_error(grp_h1['y'], hexp_h1)
mse_hexp_10 = mean_squared_error(grp_h10['y'], hexp_h10)
mse_roll_1 = mean_squared_error(grp_h1['y'], roll_h1)
mse_roll_10 = mean_squared_error(grp_h10['y'], roll_h10)

print(f"Baseline T0 h=1  | Hist_Exp MSE: {mse_hexp_1:.6e} (Target: 2.689570e-06) | Roll_20 MSE: {mse_roll_1:.6e} (Target: 2.407078e-06)")
print(f"Baseline T0 h=10 | Hist_Exp MSE: {mse_hexp_10:.6e} (Target: 2.686277e-06) | Roll_20 MSE: {mse_roll_10:.6e} (Target: 2.555697e-06)")

assert abs(mse_hexp_1 - 2.689570e-06) < 1e-10, "MSE Hist_Exp h=1 tidak cocok dengan 2h!"
assert abs(mse_hexp_10 - 2.686277e-06) < 1e-10, "MSE Hist_Exp h=10 tidak cocok dengan 2h!"
assert abs(mse_roll_1 - 2.407078e-06) < 1e-10, "MSE Roll_20 h=1 tidak cocok dengan 2h!"
assert abs(mse_roll_10 - 2.555697e-06) < 1e-10, "MSE Roll_20 h=10 tidak cocok dengan 2h!"
print("✅ Seluruh nilai baseline Hist_Exp & Roll_20 identik sempurna dengan Checkpoint 2h/3b!")

# 3. P2: Benchmark HAR v2 & Strict Assert (Tanpa try/except)
har_h1_file = os.path.join(DATASET_DIR, 'har_preds_T0_h1_v2.csv')
har_h10_file = os.path.join(DATASET_DIR, 'har_preds_T0_h10_v2.csv')
assert os.path.exists(har_h1_file), f"File {har_h1_file} tidak ditemukan!"
assert os.path.exists(har_h10_file), f"File {har_h10_file} tidak ditemukan!"

df_har1 = pd.read_csv(har_h1_file)
df_har10 = pd.read_csv(har_h10_file)

mse_har_1 = mean_squared_error(df_har1['True'], df_har1['HAR'])
mse_har_10 = mean_squared_error(df_har10['True'], df_har10['HAR'])
print(f"HAR v2 T0 h=1  MSE: {mse_har_1:.7e} (Target: 2.2346806e-06)")
print(f"HAR v2 T0 h=10 MSE: {mse_har_10:.7e} (Target: 2.4397431e-06)")
assert abs(mse_har_1 - 2.2346806e-06) < 1e-8, f"ASSERT STOP: MSE HAR h=1 {mse_har_1:.7e} != 2.2347e-06"
assert abs(mse_har_10 - 2.4397431e-06) < 1e-8, f"ASSERT STOP: MSE HAR h=10 {mse_har_10:.7e} != 2.4397e-06"

# P2b: Uji Diebold-Mariano HAR v2 vs Baselines (Strict Assert Menghentikan Eksekusi)
# Sumber target: rsic-v5-inputs/har_preds_T0_h1_v2.csv (MD5 e340a7ba8effd60c8c6d7b85b073aded)
# dan rsic-v5-inputs/har_preds_T0_h10_v2.csv (MD5 43baf149f5f60be52a80b962ab93e481)
USE_HLN = True

# DM T0 h=1: HAR vs Hist_Exp (Target: stat -3.0651, p_raw 0.0022, p_bonf 0.0174)
res_dm_h1 = v5_helpers.diebold_mariano_test(df_har1['True'], df_har1['HAR'], df_har1['Hist_Exp'], h=1, m_family=8, use_hln=USE_HLN)
print(f"PASS: DM T0 h=1 HAR vs Hist_Exp | stat: {res_dm_h1['stat']:.4f} | p_raw: {res_dm_h1['p_raw']:.4f} | p_bonf: {res_dm_h1['p_bonf']:.4f}")
assert abs(res_dm_h1['stat'] - (-3.0651)) <= 1e-3, f"ASSERT STOP: DM T0 h=1 stat {res_dm_h1['stat']:.4f} != -3.0651"
assert abs(res_dm_h1['p_raw'] - 0.0022) <= 1e-4, f"ASSERT STOP: DM T0 h=1 p_raw {res_dm_h1['p_raw']:.4f} != 0.0022"

# DM T0 h=1: HAR vs Roll_20 (Target: stat -1.4992, p_raw 0.1338)
res_dm_roll1 = v5_helpers.diebold_mariano_test(df_har1['True'], df_har1['HAR'], df_har1['Roll_20'], h=1, m_family=8, use_hln=USE_HLN)
print(f"PASS: DM T0 h=1 HAR vs Roll_20  | stat: {res_dm_roll1['stat']:.4f} | p_raw: {res_dm_roll1['p_raw']:.4f} | p_bonf: {res_dm_roll1['p_bonf']:.4f}")
assert abs(res_dm_roll1['stat'] - (-1.4992)) <= 1e-3, f"ASSERT STOP: DM T0 h=1 Roll_20 stat {res_dm_roll1['stat']:.4f} != -1.4992"
assert abs(res_dm_roll1['p_raw'] - 0.1338) <= 1e-4, f"ASSERT STOP: DM T0 h=1 Roll_20 p_raw {res_dm_roll1['p_raw']:.4f} != 0.1338"

# DM T0 h=10: HAR vs Hist_Exp (Target: stat -2.6871, p_raw 0.0072, p_bonf 0.0577)
res_dm_h10 = v5_helpers.diebold_mariano_test(df_har10['True'], df_har10['HAR'], df_har10['Hist_Exp'], h=10, m_family=8, use_hln=USE_HLN)
print(f"PASS: DM T0 h=10 HAR vs Hist_Exp | stat: {res_dm_h10['stat']:.4f} | p_raw: {res_dm_h10['p_raw']:.4f} | p_bonf: {res_dm_h10['p_bonf']:.4f}")
assert abs(res_dm_h10['stat'] - (-2.6871)) <= 1e-3, f"ASSERT STOP: DM T0 h=10 stat {res_dm_h10['stat']:.4f} != -2.6871"
assert abs(res_dm_h10['p_raw'] - 0.0072) <= 1e-4, f"ASSERT STOP: DM T0 h=10 p_raw {res_dm_h10['p_raw']:.4f} != 0.0072"

# DM T0 h=10: HAR vs Roll_20 (Target: stat -0.9228, p_raw 0.3561) (Item 5)
res_dm_roll10 = v5_helpers.diebold_mariano_test(df_har10['True'], df_har10['HAR'], df_har10['Roll_20'], h=10, m_family=8, use_hln=USE_HLN)
print(f"PASS: DM T0 h=10 HAR vs Roll_20  | stat: {res_dm_roll10['stat']:.4f} | p_raw: {res_dm_roll10['p_raw']:.4f} | p_bonf: {res_dm_roll10['p_bonf']:.4f}")
assert abs(res_dm_roll10['stat'] - (-0.9228)) <= 1e-3, f"ASSERT STOP: DM T0 h=10 Roll_20 stat {res_dm_roll10['stat']:.4f} != -0.9228"
assert abs(res_dm_roll10['p_raw'] - 0.3561) <= 1e-4, f"ASSERT STOP: DM T0 h=10 Roll_20 p_raw {res_dm_roll10['p_raw']:.4f} != 0.3561"

# P3: Uji GARCH Direct Merge
garch_file = os.path.join(DATASET_DIR, 'GARCH_Predictions_V4.csv')
assert os.path.exists(garch_file), f"ASSERT STOP: File {garch_file} tidak ditemukan di {DATASET_DIR}!"
df_garch = pd.read_csv(garch_file)
m_garch = v5_helpers.load_garch_merged(df_cv4, df_garch)
assert len(m_garch) == 3350, f"ASSERT STOP: Panjang m_garch {len(m_garch)} != 3350!"
assert 'GARCH11' in m_garch.columns and 'EGARCH' in m_garch.columns, "ASSERT STOP: Kolom GARCH11/EGARCH tidak ditemukan!"
print(f"✅ GARCH Direct Merge Validasi PASS (3350 baris, MSE h=1 2.313042e-06, MSE h=10 2.478223e-06)")

print("\n=== HASIL STAGE A: SELURUH ASSERT DATA, BENCHMARK HAR v2, & DM LULUS 100% ===")
if STAGE == "A":
    print("STAGE='A' selesai. Eksekusi berhenti di sini sesuai protokol tanpa pelatihan model GPU.")


=== STAGE A: AUDIT PROVENANCE & CROSS-CHECK BENCHMARK ===
Duplikat [cutoff, real_date] di CV_Predictions_V4: 0 (PASS: harus 0)
Baseline T0 h=1  | Hist_Exp MSE: 2.689570e-06 (Target: 2.689570e-06) | Roll_20 MSE: 2.407078e-06 (Target: 2.407078e-06)
Baseline T0 h=10 | Hist_Exp MSE: 2.686277e-06 (Target: 2.686277e-06) | Roll_20 MSE: 2.555697e-06 (Target: 2.555697e-06)
✅ Seluruh nilai baseline Hist_Exp & Roll_20 identik sempurna dengan Checkpoint 2h/3b!
HAR v2 T0 h=1  MSE: 2.2346806e-06 (Target: 2.2346806e-06)
HAR v2 T0 h=10 MSE: 2.4397431e-06 (Target: 2.4397431e-06)
PASS: DM T0 h=1 HAR vs Hist_Exp | stat: -3.0651 | p_raw: 0.0022 | p_bonf: 0.0174
PASS: DM T0 h=1 HAR vs Roll_20  | stat: -1.4992 | p_raw: 0.1338 | p_bonf: 1.0000
PASS: DM T0 h=10 HAR vs Hist_Exp | stat: -2.6871 | p_raw: 0.0072 | p_bonf: 0.0577
PASS: DM T0 h=10 HAR vs Roll_20  | stat: -0.9228 | p_raw: 0.3561 | p_bonf: 1.0000
✅ GARCH Direct Merge Validasi PASS (3350 baris, MSE h=1 2.313042e-06, MSE h=10 2.478223e-06)

=== HASIL S

# CELL 5: MODEL FACTORY FUNCTION & TABEL KONFIGURASI A PRIORI


In [5]:
# Aritmetika Early Stopping (P4):
# patience = 20 validation checks, val_check_steps = 25 steps.
# Total langkah evaluasi: 20 * 25 = 500 steps <= max_steps (1000 steps).
# Nilai dipilih a priori untuk menjamin early stopping dapat terpicu secara sah.

def get_model_instance(model_name, fset_cols, h=10, input_size=50, max_steps=1000, val_check_steps=25, patience=20, is_smoke=False):
    steps = 20 if is_smoke else max_steps
    val_steps = 10 if is_smoke else val_check_steps
    
    kwargs = {
        'h': h,
        'input_size': input_size,
        'hist_exog_list': fset_cols,
        'max_steps': steps,
        'val_check_steps': val_steps,
        'early_stop_patience_steps': patience,
        'random_seed': 42,
        'scaler_type': 'standard'
    }
    
    if model_name == 'TFT':
        return TFT(**kwargs)
    elif model_name == 'NBEATSx':
        return NBEATSx(**kwargs)
    elif model_name == 'TiDE':
        return TiDE(**kwargs)
    elif model_name == 'TSMixerx':
        kwargs['n_series'] = 1
        return TSMixerx(**kwargs)
    elif model_name == 'KAN':
        return KAN(**kwargs)
    elif model_name == 'NHITS':
        return NHITS(**kwargs)
    elif model_name == 'LSTM':
        return LSTM(**kwargs)
    else:
        raise ValueError(f"Model {model_name} tidak dikenal!")

print("=== TABEL KONFIGURASI MODEL V5S (SENSITIVITY PATIENCE=20) ===")
cfg_records = []
for (m_name, fset_name, tgt) in CONFIGS:
    fset = FEATURE_SETS[fset_name]
    cfg_records.append({
        'Model': m_name,
        'FeatureSet': fset_name,
        'N_Features': len(fset),
        'Target': tgt,
        'max_steps': MAX_STEPS,
        'val_check': VAL_CHECK_STEPS,
        'patience_checks': CHOSEN_PATIENCE,
        'patience_steps': CHOSEN_PATIENCE * VAL_CHECK_STEPS,
        'seed': SEED
    })
print(pd.DataFrame(cfg_records).to_string(index=False))


=== TABEL KONFIGURASI MODEL V5S (SENSITIVITY PATIENCE=20) ===
  Model FeatureSet  N_Features Target  max_steps  val_check  patience_checks  patience_steps  seed
    TFT      SPILL          20     T0       1000         25               20             500    42
    TFT    NOSPILL          11     T0       1000         25               20             500    42
NBEATSx      SPILL          20     T0       1000         25               20             500    42
NBEATSx    NOSPILL          11     T0       1000         25               20             500    42


# CELL 6: STAGE B & C - Training Engine & Cross Validation	


In [6]:
if STAGE in ["B", "C"]:
    is_smoke = (STAGE == "B")
    n_windows = 3 if is_smoke else n_test
    
    configs_to_run = [('TiDE', 'SPILL', 'T0')] if is_smoke else CONFIGS
    cv_predictions_dict = {}
    fit_times_smoke = []
    
    for (model_name, fset_name, target_label) in configs_to_run:
        fset_cols = FEATURE_SETS[fset_name]
        target_col = 'volatility_btc_idr' if target_label == 'T0' else 'parkinson'
        
        # P7: Hash nama file partial unik berbasis konfigurasi
        cfg_str = f"{model_name}_{fset_name}_{target_label}_in{INPUT_SIZE}_max{MAX_STEPS}_pat{CHOSEN_PATIENCE}_val{VAL_CHECK_STEPS}_seed{SEED}"
        cfg_hash = hashlib.md5(cfg_str.encode()).hexdigest()[:8]
        partial_file = os.path.join(OUTPUT_DIR, f"v5s_partial_{model_name}_{fset_name}_{target_label}_{cfg_hash}.csv")
        
        if os.path.exists(partial_file) and not is_smoke:
            print(f"  [RESUME] Menemukan checkpoint parsial {partial_file}. Melewati training...")
            cv_predictions_dict[f"{model_name}_{fset_name}_{target_label}"] = pd.read_csv(partial_file)
            continue
            
        print(f"\n--> Melatih {model_name} | FeatureSet: {fset_name} | Target: {target_label} (n_windows={n_windows})...")
        
        # Inisialisasi model
        from pytorch_lightning.loggers import CSVLogger
        csv_logger = CSVLogger(save_dir=OUTPUT_DIR, name='v5s_logs', version=f"{model_name}_{fset_name}_{target_label}")
        m = get_model_instance(
            model_name=model_name,
            fset_cols=fset_cols,
            h=H,
            input_size=INPUT_SIZE,
            max_steps=20 if is_smoke else MAX_STEPS,
            val_check_steps=10 if is_smoke else VAL_CHECK_STEPS,
            patience=CHOSEN_PATIENCE,
            is_smoke=is_smoke
        )
        
        # Item 1: Konfigurasi Trainer via .update() agar callbacks EarlyStopping tidak terhapus
        m.trainer_kwargs.update({
            'enable_progress_bar': False,
            'enable_checkpointing': False,
            'logger': csv_logger
        })
        if torch.cuda.is_available():
            m.trainer_kwargs['accelerator'] = 'cuda'
            m.trainer_kwargs['devices'] = 1
            
        # Item 1: Cetak isi trainer_kwargs['callbacks'] sebelum fit
        cbs_before = m.trainer_kwargs.get('callbacks', [])
        print(f"  [CALLBACKS SEBELUM FIT] {model_name} (isi trainer_kwargs['callbacks']): {cbs_before}")
            
        # Siapkan dataframe untuk target & fitur
        nf_df_sub = nf_df[['unique_id', 'ds', 'real_date', 'y'] + fset_cols].copy()
        if target_label == 'T2':
            nf_df_sub['y'] = df['parkinson'].values.astype('float32')
            
        df_train_val = nf_df_sub.iloc[:n_train + n_val].copy()
        
        # Item 2: Cetak checksum bobot (sum parameter) sebelum fit 1
        def get_weights_checksum(model_inst):
            return float(sum(p.detach().cpu().double().sum().item() for p in model_inst.parameters()))
            
        chk_0 = get_weights_checksum(m)
        print(f"  [BOBOT SEBELUM FIT 1] {model_name} Checksum: {chk_0:.8e}")
        
        start_t = time.time()
        nf_single = NeuralForecast(models=[m], freq='D')
        
        # Fit pertama
        if is_smoke and REFIT_PER_CUTOFF:
            for win_idx in range(3):
                t0 = time.time()
                nf_single.fit(df=df_train_val.drop(columns=['real_date']), val_size=n_val)
                fit_times_smoke.append(time.time() - t0)
        else:
            nf_single.fit(df=df_train_val.drop(columns=['real_date']), val_size=n_val)
            
        # Item 2: Cetak checksum bobot setelah fit pertama
        m_trained = nf_single.models[0]
        print(f"  [MODEL INSTANCE CHECK] nf_single.models[0] is m: {m_trained is m}")
        chk_1 = get_weights_checksum(m_trained)
        print(f"  [BOBOT SETELAH FIT 1] {model_name} Checksum: {chk_1:.8e} (Delta vs Sebelum Fit 1: {chk_1 - chk_0:.8e})")
            
        # Poin 2: Cek callback EarlyStopping terpasang pada nf_single.models[0] dari trainer_kwargs
        if is_smoke:
            cbs = m_trained.trainer_kwargs.get('callbacks', [])
            has_es = any('EarlyStopping' in str(cb) for cb in cbs)
            print(f"  [STAGE B] Callback EarlyStopping terdeteksi di trainer_kwargs: {has_es} (isi: {cbs})")
            assert has_es, "ASSERT STOP: Callback EarlyStopping tidak terpasang di trainer_kwargs!"
            
        # Cross Validation (Fit kedua via _no_refit_cross_validation)
        cv_res = nf_single.cross_validation(
            df=nf_df_sub.drop(columns=['real_date']),
            n_windows=n_windows,
            step_size=1,
            val_size=n_val,
            refit=REFIT_PER_CUTOFF
        ).reset_index(drop=True)
        
        elapsed = time.time() - start_t
        print(f"      Selesai dalam {elapsed:.2f} s ({elapsed/n_windows:.3f} s/cutoff).")
        
        # Item 2: Cetak checksum bobot setelah cross_validation (fit kedua)
        chk_2 = get_weights_checksum(m_trained)
        print(f"  [BOBOT SETELAH CROSS_VALIDATION (FIT 2)] {model_name} Checksum: {chk_2:.8e} (Delta vs Fit 1: {chk_2 - chk_1:.8e})")
        warm_start_observed = (chk_2 != chk_1)
        print(f"  [WARM START CHECK] WARM START TERAMATI: {warm_start_observed}")
        
        # Sinkronisasi tanggal nyata
        cv_res['ds'] = pd.to_datetime(cv_res['ds'])
        nf_dates = nf_df[['unique_id', 'ds', 'real_date']].copy()
        nf_dates['ds'] = pd.to_datetime(nf_dates['ds'])
        cv_res = pd.merge(cv_res, nf_dates, on=['unique_id', 'ds'], how='left')
        
        # P4: Cek Early Stopping Terpicu
        es_stopped = False
        actual_global_step = getattr(m_trained.trainer, 'global_step', None)
        if hasattr(m_trained.trainer, 'callbacks'):
            for cb in m_trained.trainer.callbacks:
                if cb.__class__.__name__ == 'EarlyStopping':
                    es_stopped = getattr(cb, 'stopped_epoch', 0) > 0
        status_es_str = "EARLY STOP TERPICU" if es_stopped else "TIDAK TERPICU"
        print(f"      Status Early Stopping: {status_es_str} (actual_step: {actual_global_step})")
        
        # Item 1: Logging State Early Stopping Internal SETELAH cross_validation (fit kedua)
        cb_obj = getattr(m_trained, '_early_stop_cb', None)
        if cb_obj is None and hasattr(m_trained, 'trainer_kwargs'):
            for cb in m_trained.trainer_kwargs.get('callbacks', []):
                if cb.__class__.__name__ == 'EarlyStopping':
                    cb_obj = cb
                    break
        stopped_epoch_val = getattr(cb_obj, 'stopped_epoch', None) if cb_obj is not None else None
        best_score_val = None
        if cb_obj is not None:
            best_val = getattr(cb_obj, 'best_score', None)
            best_score_val = float(best_val.cpu().item()) if hasattr(best_val, 'cpu') else (float(best_val) if best_val is not None else None)
        wait_count_val = getattr(cb_obj, 'wait_count', None) if cb_obj is not None else None
        patience_val = getattr(cb_obj, 'patience', CHOSEN_PATIENCE) if cb_obj is not None else CHOSEN_PATIENCE
        
        es_info = {
            'model': model_name,
            'feature_set': fset_name,
            'target': target_label,
            'stopped_epoch': stopped_epoch_val,
            'best_score': best_score_val,
            'wait_count': wait_count_val,
            'patience': patience_val
        }
        print(f"  [EARLY STOPPING AUDIT SETELAH CV] stopped_epoch: {es_info['stopped_epoch']} | best_score: {es_info['best_score']} | wait_count: {es_info['wait_count']} | patience: {es_info['patience']}")
        
        es_log_file = os.path.join(OUTPUT_DIR, "v5s_es_log.csv")
        df_es_log = pd.read_csv(es_log_file) if os.path.exists(es_log_file) else pd.DataFrame()
        df_es_log = pd.concat([df_es_log, pd.DataFrame([es_info])], ignore_index=True).drop_duplicates(subset=['model', 'feature_set', 'target'], keep='last')
        df_es_log.to_csv(es_log_file, index=False)
        
        # Item 1: Blok val-curve SETELAH cross_validation (fit kedua), tambah fit_id dan cetak audit per fit_id
        valcurve_dst = os.path.join(OUTPUT_DIR, f"v5s_valcurve_{model_name}_{fset_name}.csv")
        logger_obj = m_trained.trainer_kwargs.get('logger', None)
        logger_log_dir = getattr(logger_obj, 'log_dir', None) if logger_obj else (getattr(csv_logger, 'log_dir', None) if csv_logger else None)
        metrics_src = os.path.join(logger_log_dir, "metrics.csv") if logger_log_dir else None
        print(f"  [VAL CURVE PATH] m_trained.trainer_kwargs['logger'].log_dir: {logger_log_dir} (metrics_src: {metrics_src})")
        
        cfg_tag = f"{model_name}_{fset_name}_{target_label}"
        if not metrics_src or not os.path.exists(metrics_src):
            # Fallback glob HARUS mengandung nama config dan diurutkan menurut mtime
            candidates = glob.glob(os.path.join(OUTPUT_DIR, "v5s_logs", "**", f"*{cfg_tag}*", "**", "metrics.csv"), recursive=True)
            if not candidates:
                candidates = glob.glob(os.path.join(OUTPUT_DIR, "v5s_logs", f"*{cfg_tag}*", "metrics.csv"))
            if candidates:
                candidates = sorted(candidates, key=lambda p: os.path.getmtime(p))
                metrics_src = candidates[-1]
                print(f"  [VAL CURVE FALLBACK] Menemukan file metrics melalui glob mtime: {metrics_src}")
                
        if not metrics_src or not os.path.exists(metrics_src):
            print("  [VAL CURVE] VALCURVE TIDAK DITEMUKAN")
        else:
            df_curv = pd.read_csv(metrics_src)
            # Tandai fit di metrics.csv: tambah kolom 'fit_id' (tambah 1 setiap 'step' turun terhadap baris sebelumnya)
            if 'step' in df_curv.columns:
                step_s = df_curv['step'].ffill().fillna(0)
                is_step_drop = (step_s < step_s.shift(1)).fillna(False)
                df_curv['fit_id'] = is_step_drop.cumsum().astype(int) + 1
            else:
                df_curv['fit_id'] = 1
                
            df_curv.to_csv(valcurve_dst, index=False)
            print(f"  [VAL CURVE] Berhasil disimpan dengan kolom fit_id ke: {valcurve_dst}")
            
            # Cetak per fit_id: jumlah check, step val_loss minimum, step terakhir (= step berhenti), dan stopped_epoch dari callback
            val_col = next((c for c in ['valid_loss', 'val_loss', 'ptl/val_loss'] if c in df_curv.columns and df_curv[c].dropna().count() > 0), None)
            for fid in sorted(df_curv['fit_id'].unique()):
                sub = df_curv[df_curv['fit_id'] == fid]
                n_checks = sub[val_col].dropna().count() if val_col else 0
                min_step = 'N/A'
                min_val = 'N/A'
                if val_col and n_checks > 0:
                    sub_val = sub.dropna(subset=[val_col])
                    min_idx = sub_val[val_col].idxmin()
                    min_step = sub_val.loc[min_idx, 'step'] if 'step' in sub.columns else 'N/A'
                    min_val = f"{sub_val.loc[min_idx, val_col]:.6e}"
                last_step = sub['step'].dropna().max() if 'step' in sub.columns else 'N/A'
                print(f"    * fit_id={fid}: jumlah_checks={n_checks} | min_val_loss_step={min_step} ({val_col}={min_val}) | step_berhenti={last_step} | callback_stopped_epoch={stopped_epoch_val}")
        
        # Poin 2 & 3: Ekstrak TFT importance dari nf_single.models[0] di dalam loop tepat setelah cross_validation untuk (TFT, SPILL, T0)
        if model_name == 'TFT' and fset_name == 'SPILL' and target_label == 'T0' and not is_smoke:
            print("  [TFT IMPORTANCE] Memicu forward pass/predict untuk mengisi interpretability_params...")
            try:
                _ = nf_single.predict(df=df_train_val.drop(columns=['real_date']))
                if hasattr(m_trained, 'feature_importances') and callable(getattr(m_trained, 'feature_importances')):
                    imp_dict = m_trained.feature_importances()
                    dict_keys = list(imp_dict.keys()) if isinstance(imp_dict, dict) else []
                    print(f"  [TFT IMPORTANCE SOURCE 3.3.0] Keys dict feature_importances(): {dict_keys}")
                    if isinstance(imp_dict, dict) and 'Past variable importance over time' in imp_dict:
                        df_raw_imp = imp_dict['Past variable importance over time']
                        mean_imp = df_raw_imp.mean().reset_index()
                        mean_imp.columns = ['feature', 'mean_importance']
                        df_imp = mean_imp[~mean_imp['feature'].isin(['unique_id', 'ds', 'y', 'observed_target', 'repeated_target'])].copy()
                        df_imp['n_cutoffs'] = 1  # 1 window batch evaluasi train+val
                        df_imp['extraction_path'] = 'nf_single.models[0].feature_importances()["Past variable importance over time"]'
                        df_imp['window_represented'] = 'window_terakhir_train_val_input_size_50'
                        df_imp = df_imp.sort_values(by='mean_importance', ascending=False).reset_index(drop=True)
                        imp_csv = os.path.join(OUTPUT_DIR, "v5s_tft_feature_importance.csv")
                        df_imp.to_csv(imp_csv, index=False)
                        print(f"  ✅ TFT Feature Importance berhasil diekstrak ({len(df_imp)} fitur, n_cutoffs=1) dan disimpan ke: {imp_csv}")
            except Exception as e_imp:
                print(f"  ⚠️ Ekstraksi TFT Importance: {e_imp}")
        
        # Item 4: Stage B Kriteria Lolos Numerik (Strict Assert)
        if is_smoke:
            print("\n=== STAGE B: VALIDASI NUMERIK & STRICT ASSERT (Item 4) ===")
            assert len(cv_res) == 30, f"ASSERT STOP: Panjang prediksi {len(cv_res)} != 30 (3 cutoff x 10 horizon)!"
            assert not cv_res[model_name].isna().any(), f"ASSERT STOP: Ada nilai NaN pada prediksi {model_name}!"
            pred_std = float(np.std(cv_res[model_name]))
            assert pred_std > 0, f"ASSERT STOP: Standar deviasi prediksi = {pred_std} (tidak ada variansi)!"
            
            last_30_cv4 = df_cv4.iloc[-30:].copy()
            assert (pd.to_datetime(cv_res['ds']).values == pd.to_datetime(last_30_cv4['ds']).values).all(), "ASSERT STOP: ds tidak cocok dengan 3 cutoff terakhir CV_Predictions_V4.csv!"
            assert (pd.to_datetime(cv_res['cutoff']).values == pd.to_datetime(last_30_cv4['cutoff']).values).all(), "ASSERT STOP: cutoff tidak cocok dengan 3 cutoff terakhir CV_Predictions_V4.csv!"
            assert (pd.to_datetime(cv_res['real_date']).values == pd.to_datetime(last_30_cv4['real_date']).values).all(), "ASSERT STOP: real_date tidak cocok dengan 3 cutoff terakhir CV_Predictions_V4.csv!"
            
            required_cols = ['unique_id', 'ds', 'cutoff', model_name, 'y', 'real_date']
            assert all(c in cv_res.columns for c in required_cols), f"ASSERT STOP: Kolom tidak lengkap di cv_res!"
            assert all(c in df_cv4.columns for c in cv_res.columns), f"ASSERT STOP: Kolom cv_res {list(cv_res.columns)} bukan subset dari CV_Predictions_V4 {list(df_cv4.columns)}!"
            print(f"✅ Strict Asserts Stage B Lolos 100%: 30 baris, bebas NaN, std={pred_std:.6e} > 0, dates & columns match CV_Predictions_V4.")
            
            time_per_fit = (fit_times_smoke[1] + fit_times_smoke[2])/2.0 if len(fit_times_smoke) >= 3 else (elapsed * 0.8 / 3.0)
            time_per_step = time_per_fit / 20.0
            hours_per_cfg = ((time_per_step * MAX_STEPS * 1) + (335 * 0.1)) / 3600.0
            total_hours_default = hours_per_cfg * len(CONFIGS)
            print(f"\nEstimasi Jam GPU:")
            print(f"  - Waktu per step (non-warmup): {time_per_step*1000:.2f} ms/step")
            print(f"  - Estimasi per konfigurasi (335 cutoff, max_steps={MAX_STEPS}): ~{hours_per_cfg:.2f} jam")
            print(f"  - Total estimasi untuk daftar default ({len(CONFIGS)} konfigurasi): ~{total_hours_default:.2f} jam")
            print("Stage B selesai dengan sukses.")
        else:
            cv_res.to_csv(partial_file, index=False)
            cv_predictions_dict[f"{model_name}_{fset_name}_{target_label}"] = cv_res
            zip_saved = save_zip_backup()
            print(f"      Tersimpan checkpoint: {partial_file} & zip backup: {zip_saved}")




Seed set to 42



--> Melatih TFT | FeatureSet: SPILL | Target: T0 (n_windows=335)...
  [CALLBACKS SEBELUM FIT] TFT (isi trainer_kwargs['callbacks']): [<pytorch_lightning.callbacks.early_stopping.EarlyStopping object at 0x7ba886578ad0>]
  [BOBOT SEBELUM FIT 1] TFT Checksum: 3.51943501e+03


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                    ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                    │ MAE                      │      0 │ train │     0 │
│ 1 │ hist_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 2 │ futr_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 3 │ stat_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 4 │ padder_train            │ ConstantPad1d            │      0 │ train │     0 │
│ 5 │ scaler                  │ TemporalNorm             │      0 │ train │     0 │
│ 6 │ embedding               │ TFTEmbedding             │  5.6 K │ train │     0 │
│ 7 │ temporal_encoder        │ TemporalCovariateEncoder │  2.3 M │ train │     0 │
│ 8 │ temporal_fusion_decoder │ TemporalFusionDecoder    │  256 K │ train │     0 │
│ 9 │ output_adapter          │ Linear                   │    129 │ train │     0 │
└───┴─────────────────────────┴──────────────────────────┴────────┴───────┴───────┘

Trainable params: 2.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.6 M                                                                                                
Total estimated model params size (MB): 10.383                                                                     
Modules in train mode: 251                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

  [MODEL INSTANCE CHECK] nf_single.models[0] is m: False
  [BOBOT SETELAH FIT 1] TFT Checksum: 3.78726110e+03 (Delta vs Sebelum Fit 1: 2.67826093e+02)


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                    ┃ Type                     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                    │ MAE                      │      0 │ train │     0 │
│ 1 │ hist_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 2 │ futr_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 3 │ stat_cat_embeddings     │ ModuleList               │      0 │ train │     0 │
│ 4 │ padder_train            │ ConstantPad1d            │      0 │ train │     0 │
│ 5 │ scaler                  │ TemporalNorm             │      0 │ train │     0 │
│ 6 │ embedding               │ TFTEmbedding             │  5.6 K │ train │     0 │
│ 7 │ temporal_encoder        │ TemporalCovariateEncoder │  2.3 M │ train │     0 │
│ 8 │ temporal_fusion_decoder │ TemporalFusionDecoder    │  256 K │ train │     0 │
│ 9 │ output_adapter          │ Linear                   │    129 │ train │     0 │
└───┴─────────────────────────┴──────────────────────────┴────────┴───────┴───────┘

... (20 outputs left out)

  [MODEL INSTANCE CHECK] nf_single.models[0] is m: False
  [BOBOT SETELAH FIT 1] NBEATSx Checksum: -2.99557066e+03 (Delta vs Sebelum Fit 1: -3.12960862e+03)


┏━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                │ MAE           │      0 │ train │     0 │
│ 1 │ hist_cat_embeddings │ ModuleList    │      0 │ train │     0 │
│ 2 │ futr_cat_embeddings │ ModuleList    │      0 │ train │     0 │
│ 3 │ stat_cat_embeddings │ ModuleList    │      0 │ train │     0 │
│ 4 │ padder_train        │ ConstantPad1d │      0 │ train │     0 │
│ 5 │ scaler              │ TemporalNorm  │      0 │ train │     0 │
│ 6 │ blocks              │ ModuleList    │  3.3 M │ train │     0 │
└───┴─────────────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 3.3 M                                                                                            
Non-trainable params: 1.3 K                                                                                        
Total params: 3.3 M                                                                                                
Total estimated model params size (MB): 13.363                                                                     
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

      Selesai dalam 12.69 s (0.038 s/cutoff).
  [BOBOT SETELAH CROSS_VALIDATION (FIT 2)] NBEATSx Checksum: -4.88719138e+03 (Delta vs Fit 1: -1.89162072e+03)
  [WARM START CHECK] WARM START TERAMATI: True
      Status Early Stopping: EARLY STOP TERPICU (actual_step: 0)
  [EARLY STOPPING AUDIT SETELAH CV] stopped_epoch: 549 | best_score: 0.000925573636777699 | wait_count: 20 | patience: 20
  [VAL CURVE PATH] m_trained.trainer_kwargs['logger'].log_dir: /kaggle/working/v5s_logs/NBEATSx_NOSPILL_T0 (metrics_src: /kaggle/working/v5s_logs/NBEATSx_NOSPILL_T0/metrics.csv)
  [VAL CURVE] Berhasil disimpan dengan kolom fit_id ke: /kaggle/working/v5s_valcurve_NBEATSx_NOSPILL.csv
    * fit_id=1: jumlah_checks=21 | min_val_loss_step=24 (valid_loss=8.321030e-04) | step_berhenti=524 | callback_stopped_epoch=549
    * fit_id=2: jumlah_checks=22 | min_val_loss_step=49 (valid_loss=9.255736e-04) | step_berhenti=549 | callback_stopped_epoch=549
      Tersimpan checkpoint: /kaggle/working/v5s_partial_NBEATSx_

# CELL 7: METRIK EVALUASI MULTI-HORIZON & ABLATION (MSE, MAE, R2_OOS)


In [7]:
if STAGE == "C":
    import v5_helpers
    all_metrics_list = []
    
    # Item 7: Sertakan baris HAR, GARCH11, EGARCH, Hist_Exp, Roll_20 di tabel metrik
    har_h1_file = os.path.join(DATASET_DIR, "har_preds_T0_h1_v2.csv")
    har_h10_file = os.path.join(DATASET_DIR, "har_preds_T0_h10_v2.csv")
    garch_file = os.path.join(DATASET_DIR, "GARCH_Predictions_V4.csv")
    assert os.path.exists(garch_file), f"ASSERT STOP: File {garch_file} tidak ditemukan!"
    
    if os.path.exists(har_h1_file) and os.path.exists(har_h10_file):
        df_h1_b = pd.read_csv(har_h1_file)
        df_h10_b = pd.read_csv(har_h10_file)
        for h_step, df_hb in [('h=1', df_h1_b), ('h=10', df_h10_b)]:
            y_tr = df_hb['True'].values
            hexp_v = df_hb['Hist_Exp'].values
            roll_v = df_hb['Roll_20'].values
            
            for b_col in ['Hist_Exp', 'Roll_20', 'HAR']:
                m_dict = v5_helpers.evaluate_metrics(y_tr, df_hb[b_col].values, hexp_v, roll_v)
                m_dict['Config'] = f"{b_col}_Baseline"
                m_dict['Model'] = b_col
                m_dict['FeatureSet'] = '-'
                m_dict['Target'] = 'T0'
                m_dict['Horizon'] = h_step
                all_metrics_list.append(m_dict)
                
    df_garch_raw = pd.read_csv(garch_file)
    m_garch_df = v5_helpers.load_garch_merged(df_cv4, df_garch_raw)
    for h_val, h_step in [(1, 'h=1'), (10, 'h=10')]:
        sub_g = m_garch_df[m_garch_df['h_step'] == h_val]
        y_tr = sub_g['y'].values
        df_hb_ref = pd.read_csv(har_h1_file if h_val == 1 else har_h10_file)
        hexp_v = df_hb_ref['Hist_Exp'].values
        roll_v = df_hb_ref['Roll_20'].values
        
        for g_col in ['GARCH11', 'EGARCH']:
            m_dict = v5_helpers.evaluate_metrics(y_tr, sub_g[g_col].values, hexp_v, roll_v)
            m_dict['Config'] = f"{g_col}_Baseline"
            m_dict['Model'] = g_col
            m_dict['FeatureSet'] = '-'
            m_dict['Target'] = 'T0'
            m_dict['Horizon'] = h_step
            all_metrics_list.append(m_dict)
    
    # Metrik Model Deep Learning
    for cfg_key, cv_df_t in cv_predictions_dict.items():
        m_name, fset_name, target_label = cfg_key.split('_')
        hist_s = hist_series if target_label == 'T0' else pd.Series(df['parkinson'].values, index=df['Date'])
        c_origins = v5_helpers.get_cutoff_origins(cv_df_t, hist_s)
        
        for h_step, grp in [('h=1', cv_df_t.groupby('cutoff').head(1)), ('h=10', cv_df_t.groupby('cutoff').tail(1))]:
            y_true = grp['y'].values
            hexp_arr, roll_arr, _ = v5_helpers.compute_baselines(grp, hist_s, c_origins)
            
            if m_name in grp.columns:
                m_preds = grp[m_name].values
                m_dict = v5_helpers.evaluate_metrics(y_true, m_preds, hexp_arr, roll_arr)
                m_dict['Config'] = cfg_key
                m_dict['Model'] = m_name
                m_dict['FeatureSet'] = fset_name
                m_dict['Target'] = target_label
                m_dict['Horizon'] = h_step
                all_metrics_list.append(m_dict)

    df_metrics = pd.DataFrame(all_metrics_list)
    metrics_csv = os.path.join(OUTPUT_DIR, "v5s_metrics_table.csv")
    df_metrics.to_csv(metrics_csv, index=False)
    save_zip_backup()
    print("\n=== TABEL METRIK EVALUASI KOMPREHENSIF V5S (SENSITIVITY PATIENCE=20) ===")
    cols_show = ['Config', 'Model', 'Horizon', 'MSE', 'MAE', 'R2_OOS_HistExp', 'R2_OOS_Roll20', 'Pred_Std', 'Pred_Corr']
    if not df_metrics.empty:
        print(df_metrics[cols_show].to_string(index=False))
    print(f"Tersimpan ke: {metrics_csv}")



=== TABEL METRIK EVALUASI KOMPREHENSIF V5S (SENSITIVITY PATIENCE=20) ===
            Config    Model Horizon      MSE      MAE  R2_OOS_HistExp  R2_OOS_Roll20  Pred_Std  Pred_Corr
 Hist_Exp_Baseline Hist_Exp     h=1 0.000003 0.000967        0.000000      -0.117264  0.000034  -0.109538
  Roll_20_Baseline  Roll_20     h=1 0.000002 0.000567        0.104957       0.000000  0.000504   0.139280
      HAR_Baseline      HAR     h=1 0.000002 0.000672        0.169061       0.071621  0.000246   0.300802
 Hist_Exp_Baseline Hist_Exp    h=10 0.000003 0.000964        0.000000      -0.051006  0.000034  -0.107244
  Roll_20_Baseline  Roll_20    h=10 0.000003 0.000587        0.048531       0.000000  0.000504   0.045972
      HAR_Baseline      HAR    h=10 0.000002 0.000719        0.091699       0.045371  0.000117   0.008606
  GARCH11_Baseline  GARCH11     h=1 0.000002 0.000650        0.139923       0.039066  0.000457   0.202289
   EGARCH_Baseline   EGARCH     h=1 0.000002 0.000648        0.160718       0.

# CELL 8: UJI SIGNIFIKANSI STATISTIK DIEBOLD-MARIANO (BONFERRONI & ABLATION)


In [8]:
if STAGE == "C":
    import v5_helpers
    dm_results_list = []
    
    har_h1_file = os.path.join(DATASET_DIR, "har_preds_T0_h1_v2.csv")
    har_h10_file = os.path.join(DATASET_DIR, "har_preds_T0_h10_v2.csv")
    df_har_preds = {}
    if os.path.exists(har_h1_file) and os.path.exists(har_h10_file):
        df_har_preds['h=1'] = pd.read_csv(har_h1_file)
        df_har_preds['h=10'] = pd.read_csv(har_h10_file)
        
    # 1. Komparasi Model DL vs Baselines (Hist_Exp, Roll_20, HAR)
    for cfg_key, cv_df_t in cv_predictions_dict.items():
        m_name, fset_name, target_label = cfg_key.split('_')
        hist_s = hist_series if target_label == 'T0' else pd.Series(df['parkinson'].values, index=df['Date'])
        c_origins = v5_helpers.get_cutoff_origins(cv_df_t, hist_s)
        
        for h_step, grp in [('h=1', cv_df_t.groupby('cutoff').head(1)), ('h=10', cv_df_t.groupby('cutoff').tail(1))]:
            h_val = 1 if h_step == 'h=1' else 10
            y_true = grp['y'].values
            hexp_arr, roll_arr, _ = v5_helpers.compute_baselines(grp, hist_s, c_origins)
            
            baselines_dict = {'Hist_Exp': hexp_arr, 'Roll_20': roll_arr}
            if target_label == 'T0' and h_step in df_har_preds and len(df_har_preds[h_step]) == len(y_true):
                baselines_dict['HAR'] = df_har_preds[h_step]['HAR'].values
                
            m_family = len(CONFIGS) * len(baselines_dict)
            if m_name in grp.columns:
                m_preds = grp[m_name].values
                for b_name, b_preds in baselines_dict.items():
                    res_dm = v5_helpers.diebold_mariano_test(y_true, m_preds, b_preds, h=h_val, m_family=m_family, use_hln=True)
                    res_dm['Comparison'] = f"{cfg_key} vs {b_name}"
                    res_dm['Model'] = m_name
                    res_dm['Config'] = cfg_key
                    res_dm['Horizon'] = h_step
                    res_dm['Benchmark'] = b_name
                    res_dm['m_family'] = m_family
                    res_dm['Winner'] = cfg_key if res_dm['p_bonf'] < 0.05 and res_dm['md_MSE'] < 0 else (b_name if res_dm['p_bonf'] < 0.05 and res_dm['md_MSE'] > 0 else 'Tie')
                    dm_results_list.append(res_dm)

    print("\n=== UJI DIEBOLD-MARIANO ABLATION LANGSUNG: SPILL vs NOSPILL & SPILL vs VOLX ===")
    for model_target in ['TFT', 'NBEATSx']:
        for h_step in ['h=1', 'h=10']:
            h_val = 1 if h_step == 'h=1' else 10
            spill_k = f"{model_target}_SPILL_T0"
            nospill_k = f"{model_target}_NOSPILL_T0"
            volx_k = f"{model_target}_VOLX_T0"
            
            if spill_k in cv_predictions_dict and nospill_k in cv_predictions_dict:
                grp_sp = cv_predictions_dict[spill_k].groupby('cutoff').head(1) if h_val == 1 else cv_predictions_dict[spill_k].groupby('cutoff').tail(1)
                grp_no = cv_predictions_dict[nospill_k].groupby('cutoff').head(1) if h_val == 1 else cv_predictions_dict[nospill_k].groupby('cutoff').tail(1)
                
                y_tr = grp_sp['y'].values
                pred_sp = grp_sp[model_target].values
                pred_no = grp_no[model_target].values
                
                res_ab1 = v5_helpers.diebold_mariano_test(y_tr, pred_sp, pred_no, h=h_val, m_family=8, use_hln=True)
                res_ab1['Comparison'] = f"{model_target} SPILL vs NOSPILL"
                res_ab1['Model'] = model_target
                res_ab1['Config'] = spill_k
                res_ab1['Horizon'] = h_step
                res_ab1['Benchmark'] = nospill_k
                res_ab1['m_family'] = 8
                res_ab1['Winner'] = 'SPILL' if res_ab1['p_bonf'] < 0.05 and res_ab1['md_MSE'] < 0 else ('NOSPILL' if res_ab1['p_bonf'] < 0.05 and res_ab1['md_MSE'] > 0 else 'Tie')
                dm_results_list.append(res_ab1)
                
            if spill_k in cv_predictions_dict and volx_k in cv_predictions_dict:
                grp_vx = cv_predictions_dict[volx_k].groupby('cutoff').head(1) if h_val == 1 else cv_predictions_dict[volx_k].groupby('cutoff').tail(1)
                pred_vx = grp_vx[model_target].values
                
                res_ab2 = v5_helpers.diebold_mariano_test(y_tr, pred_sp, pred_vx, h=h_val, m_family=8, use_hln=True)
                res_ab2['Comparison'] = f"{model_target} SPILL vs VOLX"
                res_ab2['Model'] = model_target
                res_ab2['Config'] = spill_k
                res_ab2['Horizon'] = h_step
                res_ab2['Benchmark'] = volx_k
                res_ab2['m_family'] = 8
                res_ab2['Winner'] = 'SPILL' if res_ab2['p_bonf'] < 0.05 and res_ab2['md_MSE'] < 0 else ('VOLX' if res_ab2['p_bonf'] < 0.05 and res_ab2['md_MSE'] > 0 else 'Tie')
                dm_results_list.append(res_ab2)

    df_dm = pd.DataFrame(dm_results_list)
    dm_csv = os.path.join(OUTPUT_DIR, "v5s_dm_table.csv")
    df_dm.to_csv(dm_csv, index=False)
    save_zip_backup()
    print("\n=== TABEL HASIL UJI DIEBOLD-MARIANO V5S ===")
    if not df_dm.empty:
        cols_dm_disp = ['Comparison', 'Model', 'Horizon', 'md_MSE', 'stat', 'p_raw', 'p_bonf', 'm_family', 'Winner']
        print(df_dm[cols_dm_disp].to_string(index=False))
    print(f"Tersimpan ke: {dm_csv}")



=== UJI DIEBOLD-MARIANO ABLATION LANGSUNG: SPILL vs NOSPILL & SPILL vs VOLX ===

=== TABEL HASIL UJI DIEBOLD-MARIANO V5S ===
                    Comparison   Model Horizon        md_MSE      stat    p_raw   p_bonf  m_family Winner
      TFT_SPILL_T0 vs Hist_Exp     TFT     h=1  1.561283e-06  1.638708 0.101274 1.000000        12    Tie
       TFT_SPILL_T0 vs Roll_20     TFT     h=1  1.843775e-06  1.963515 0.049586 0.595036        12    Tie
           TFT_SPILL_T0 vs HAR     TFT     h=1  2.016172e-06  2.072794 0.038191 0.458297        12    Tie
      TFT_SPILL_T0 vs Hist_Exp     TFT    h=10 -4.916521e-08 -0.243627 0.807520 1.000000        12    Tie
       TFT_SPILL_T0 vs Roll_20     TFT    h=10  8.141528e-08  1.051449 0.293052 1.000000        12    Tie
           TFT_SPILL_T0 vs HAR     TFT    h=10  1.973687e-07  1.635585 0.101926 1.000000        12    Tie
    TFT_NOSPILL_T0 vs Hist_Exp     TFT     h=1  2.673214e-07  1.117563 0.263754 1.000000        12    Tie
     TFT_NOSPILL_T0 vs Rol

# CELL 9: EKSTRAKSI TFT FEATURE IMPORTANCE (P8 - TANPA FALLBACK)


In [9]:
if STAGE == "C":
    print("=== EKSTRAKSI TFT FEATURE IMPORTANCE TRANSPARAN (SENSITIVITY V5S) ===")
    imp_csv = os.path.join(OUTPUT_DIR, "v5s_tft_feature_importance.csv")
    
    if os.path.exists(imp_csv) and os.path.getsize(imp_csv) > 50:
        df_imp_read = pd.read_csv(imp_csv)
        df_imp_read = df_imp_read.sort_values(by='mean_importance', ascending=False).reset_index(drop=True)
        print(f"✅ TFT Feature Importance berhasil dimuat ({len(df_imp_read)} fitur):")
        print(df_imp_read.to_string(index=False))
        tot_imp = df_imp_read['mean_importance'].sum()
        print(f"\nTotal Mean Importance (seluruh {len(df_imp_read)} fitur): {tot_imp:.6f}")
        print("Label Metadata Representasi: window terakhir train+val, input_size=50, n_cutoffs=1")
    else:
        print(f"ℹ️ File {imp_csv} belum tersedia atau kosong. Ekstraksi dilakukan otomatis di Cell 6 saat pelatihan TFT_SPILL_T0 selesai.")
        
    save_zip_backup()


=== EKSTRAKSI TFT FEATURE IMPORTANCE TRANSPARAN (SENSITIVITY V5S) ===
✅ TFT Feature Importance berhasil dimuat (20 fitur):
                      feature  mean_importance  n_cutoffs                                                                 extraction_path                      window_represented
                      tvp_tsi         0.226171          1 nf_single.models[0].feature_importances()["Past variable importance over time"] window_terakhir_train_val_input_size_50
           ret_crypto_eth_usd         0.098920          1 nf_single.models[0].feature_importances()["Past variable importance over time"] window_terakhir_train_val_input_size_50
         net_spill_saham_msft         0.096894          1 nf_single.models[0].feature_importances()["Past variable importance over time"] window_terakhir_train_val_input_size_50
     net_spill_crypto_btc_usd         0.094969          1 nf_single.models[0].feature_importances()["Past variable importance over time"] window_terakhir_train_val_i

# CELL 10: MANIFEST OUTPUT, PERHITUNGAN MD5, & PENUTUPAN EKSEKUSI


In [10]:
print("\n=======================================================")
print("PENGECEKAN FILE OUTPUT FINAL & PERHITUNGAN MD5")
print("=======================================================")

# Update zip backup terakhir
final_zip = save_zip_backup()
print(f"✅ Backup ZIP tersimpan ke: {final_zip}")

manifest_lines = []
md5_dict = {}

for fn in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, fn)
    if os.path.isfile(fp) and fn.startswith('v5s_'):
        h = get_file_md5(fp)
        sz = os.path.getsize(fp)
        manifest_lines.append(f"{fn:<36} | Size: {sz:<10} | MD5: {h}")
        md5_dict[fn] = h
        print(f"  - {fn:<36} | {sz:>10} bytes | MD5: {h}")

manifest_file = os.path.join(OUTPUT_DIR, "v5s_md5_manifest.txt")
with open(manifest_file, "w") as f:
    f.write("\n".join(manifest_lines) + "\n")

md5_manifest_h = get_file_md5(manifest_file)
print(f"\nManifest tersimpan ke: {manifest_file} (MD5: {md5_manifest_h})")

md5_summary_str = ", ".join([f"{k}: {v}" for k, v in md5_dict.items()])
print(f"\n=== SELESAI ===")
print(f"SELESAI V5S + md5: {md5_summary_str if md5_summary_str else 'STAGE_A_COMPLETED'}")



PENGECEKAN FILE OUTPUT FINAL & PERHITUNGAN MD5
✅ Backup ZIP tersimpan ke: /kaggle/working/v5s_outputs.zip
  - v5s_dm_table.csv                     |       4275 bytes | MD5: f08219fc260bc4cc64e519da371624c1
  - v5s_es_log.csv                       |        252 bytes | MD5: fe650a7ec0f27e3504649a3857ddd8ea
  - v5s_metrics_table.csv                |       3844 bytes | MD5: c411f51ef70eb9305ead0484bdf13aa0
  - v5s_outputs.zip                      |     210421 bytes | MD5: 449eb1b322f9df310bb149ff3a113b3a
  - v5s_partial_NBEATSx_NOSPILL_T0_100e5e73.csv |     228380 bytes | MD5: a5c80c9977d59524646122cf8d9da2c8
  - v5s_partial_NBEATSx_SPILL_T0_98a23b00.csv |     228477 bytes | MD5: 5306c060d8ca306d7176e6fd02bb0b94
  - v5s_partial_TFT_NOSPILL_T0_957592e5.csv |     228308 bytes | MD5: a8f4400b3e2fb8f8cb0b451bba7ef16c
  - v5s_partial_TFT_SPILL_T0_2cd48e18.csv |     228560 bytes | MD5: 350bf118c1eb3309eb06f0b174b89f8e
  - v5s_pip_freeze.txt                   |      18131 bytes | MD5: fbba8adec4